# TP — MLP Multi-Couches MNIST | Notebook de Tests
### Licence IAIL — ML2 | 2025–2026
---
> Ce notebook importe directement les fonctions depuis `TP_MLP_multiclass_mnist_multicouches.py`.  
> Aucune fonction n'est redéfinie ici.

## 1. Import du Module

In [11]:
import numpy as np
import matplotlib.pyplot as plt

# Import de toutes les fonctions depuis le fichier Python du TP
from TP_MLP_multiclass_mnist_multicouches import (
    initialize_parameters,
    relu,
    relu_derivative,
    softmax,
    forward_propagation,
    compute_cost,
    backward_propagation,
    update_parameters,
    model,
    predict,
    one_hot,
    accuracy
)

np.random.seed(42)
print("✅ Import réussi depuis TP_MLP_multiclass_mnist_multicouches.py")

ModuleNotFoundError: No module named 'tensorflow'

## 2. Tests Unitaires

On teste chaque fonction indépendamment avec des assertions.

In [ ]:
print("=" * 55)
print("TEST 1 — initialize_parameters")
print("=" * 55)
params = initialize_parameters([784, 128, 64, 10])
assert params["W1"].shape == (128, 784)
assert params["b1"].shape == (128, 1)
assert params["W2"].shape == (64, 128)
assert params["W3"].shape == (10, 64)
print("✅ W1:", params["W1"].shape, "| b1:", params["b1"].shape)
print("✅ W2:", params["W2"].shape, "| W3:", params["W3"].shape)
print("✅ Mean abs W1 (doit être ≈ 0.01) :", round(np.mean(np.abs(params["W1"])), 5))

In [ ]:
print("=" * 55)
print("TEST 2 — relu & relu_derivative")
print("=" * 55)
Z = np.array([[-3, 0, 2, -1, 5]])
assert np.array_equal(relu(Z), [[0, 0, 2, 0, 5]])
assert np.array_equal(relu_derivative(Z), [[0, 0, 1, 0, 1]])
print("✅ relu([-3,0,2,-1,5])       =", relu(Z).flatten().tolist())
print("✅ relu_deriv([-3,0,2,-1,5]) =", relu_derivative(Z).flatten().tolist())

In [ ]:
print("=" * 55)
print("TEST 3 — softmax")
print("=" * 55)
Z_sm = np.array([[1.0], [2.0], [3.0]])
sm   = softmax(Z_sm)
assert sm.shape == (3, 1)
assert abs(np.sum(sm) - 1.0) < 1e-9,  "❌ Somme != 1"
assert np.all(sm >= 0),                "❌ Valeur négative"
print("✅ softmax([1,2,3]) =", sm.flatten().round(4).tolist())
print("✅ Somme            =", round(np.sum(sm), 10))

# Stabilité numérique
sm_big = softmax(np.array([[1000.0], [1001.0], [1002.0]]))
assert not np.any(np.isnan(sm_big)), "❌ NaN détecté !"
print("✅ Valeurs extrêmes — pas de NaN :", sm_big.flatten().round(4).tolist())

In [ ]:
print("=" * 55)
print("TEST 4 — forward_propagation")
print("=" * 55)
np.random.seed(0)
X_t  = np.random.randn(784, 5)
p_t  = initialize_parameters([784, 32, 10])
AL, cache = forward_propagation(X_t, p_t)
assert AL.shape == (10, 5),                   "❌ Shape AL incorrecte"
assert abs(np.sum(AL[:, 0]) - 1.0) < 1e-9,   "❌ Somme proba != 1"
assert np.all(AL >= 0) and np.all(AL <= 1),   "❌ AL hors [0,1]"
print("✅ Shape AL         :", AL.shape, "(attendu (10,5))")
print("✅ Somme col 0      :", round(np.sum(AL[:, 0]), 10))
print("✅ Clés du cache    :", list(cache.keys()))

In [ ]:
print("=" * 55)
print("TEST 5 — compute_cost")
print("=" * 55)
AL_ok  = np.array([[1.0], [0.0], [0.0]])
AL_bad = np.array([[0.01], [0.01], [0.98]])
Y_ref  = np.array([[1.0], [0.0], [0.0]])
c_ok  = compute_cost(AL_ok,  Y_ref)
c_bad = compute_cost(AL_bad, Y_ref)
assert c_ok < c_bad, "❌ Coût parfait devrait être < coût mauvais"
print("✅ Coût prédiction parfaite :", round(float(c_ok),  6))
print("✅ Coût mauvaise prédiction :", round(float(c_bad), 6))
print("✅ c_parfait < c_mauvais    :", c_ok < c_bad)

In [ ]:
print("=" * 55)
print("TEST 6 — backward_propagation")
print("=" * 55)
np.random.seed(1)
X_bp  = np.random.randn(784, 4)
Y_bp  = one_hot(np.array([0, 3, 7, 1]))
p_bp  = initialize_parameters([784, 16, 10])
AL_bp, cache_bp = forward_propagation(X_bp, p_bp)
grads = backward_propagation(X_bp, Y_bp, cache_bp, p_bp)
assert grads["dW1"].shape == p_bp["W1"].shape
assert grads["dW2"].shape == p_bp["W2"].shape
assert grads["db1"].shape == p_bp["b1"].shape
print("✅ dW1 shape :", grads["dW1"].shape)
print("✅ dW2 shape :", grads["dW2"].shape)
print("✅ db1 shape :", grads["db1"].shape)

In [ ]:
print("=" * 55)
print("TEST 7 — update_parameters")
print("=" * 55)
W1_before = p_bp["W1"].copy()
p_up = update_parameters(p_bp, grads, learning_rate=0.1)
assert not np.array_equal(p_up["W1"], W1_before), "❌ W1 inchangé après update"
diff = np.max(np.abs(p_up["W1"] - W1_before))
print("✅ W1 modifié — différence max :", round(diff, 8))

print()
print("=" * 55)
print("✅ TOUS LES TESTS UNITAIRES PASSÉS")
print("=" * 55)

## 3. Test d'Intégration — Mini Dataset Synthétique

On vérifie que le modèle converge bien avant de lancer MNIST.

In [ ]:
np.random.seed(42)
m, n_x, n_cls = 200, 20, 3
X_mini = np.random.randn(n_x, m)
Y_raw_mini = np.random.randint(0, n_cls, m)
Y_mini = one_hot(Y_raw_mini, num_classes=n_cls)

params_mini, costs_mini = model(
    X_mini, Y_mini,
    layers_dims=[n_x, 32, n_cls],
    num_iterations=1000,
    learning_rate=0.1,
    print_cost=True
)

assert costs_mini[-1] < costs_mini[0], "❌ Le coût n'a pas diminué !"
print(f"\n✅ Coût initial : {costs_mini[0]:.4f}  →  Coût final : {costs_mini[-1]:.4f}")
print(f"✅ Accuracy     : {accuracy(X_mini, Y_raw_mini, params_mini):.1f}%")

plt.figure(figsize=(7, 3))
plt.plot([i*100 for i in range(len(costs_mini))], costs_mini, color="#2E86C1", linewidth=2)
plt.title("Coût — Mini Dataset Synthétique")
plt.xlabel("Itérations"); plt.ylabel("Coût")
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout(); plt.show()

## 4. Chargement MNIST

In [ ]:
from tensorflow.keras.datasets import mnist

(X_train_raw, Y_train_raw), (X_test_raw, Y_test_raw) = mnist.load_data()

X_train = X_train_raw.reshape(X_train_raw.shape[0], -1).T / 255.0
X_test  = X_test_raw.reshape(X_test_raw.shape[0], -1).T  / 255.0
Y_train = one_hot(Y_train_raw)
Y_test  = one_hot(Y_test_raw)

print(f"X_train : {X_train.shape} | Y_train : {Y_train.shape}")
print(f"X_test  : {X_test.shape}  | Y_test  : {Y_test.shape}")

fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for i, ax in enumerate(axes):
    ax.imshow(X_train[:, i].reshape(28, 28), cmap="gray")
    ax.set_title(f"{Y_train_raw[i]}", fontsize=10)
    ax.axis("off")
plt.suptitle("Exemples MNIST", fontsize=11)
plt.tight_layout(); plt.show()

## 5. Entraînement — 3 Architectures

In [ ]:
architectures = [
    [784, 128, 10],
    [784, 256, 128, 10],
    [784, 512, 256, 128, 10],
]

results = {}

for arch in architectures:
    print(f"\n{'='*50}\nArchitecture : {arch}\n{'='*50}")
    params, costs = model(
        X_train, Y_train,
        layers_dims=arch,
        num_iterations=3000,
        learning_rate=0.15,
        print_cost=True
    )
    tr = accuracy(X_train, Y_train_raw, params)
    te = accuracy(X_test,  Y_test_raw,  params)
    print(f"\n  → Train : {tr:.2f}%  |  Test : {te:.2f}%")
    results[str(arch)] = {"costs": costs, "train_acc": tr, "test_acc": te, "params": params}

## 6. Visualisation des Résultats

In [ ]:
colors = ["#2E86C1", "#27AE60", "#E74C3C"]
keys   = [str(a) for a in architectures]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Courbes coût
ax = axes[0]
for k, c in zip(keys, colors):
    iters = [i*100 for i in range(len(results[k]["costs"]))]
    ax.plot(iters, results[k]["costs"], label=k, color=c, linewidth=2)
ax.set_title("Évolution du Coût", fontweight="bold")
ax.set_xlabel("Itérations"); ax.set_ylabel("Cross-Entropy")
ax.legend(fontsize=7); ax.grid(True, linestyle="--", alpha=0.5)

# Coût final
ax = axes[1]
fc   = [results[k]["costs"][-1] for k in keys]
bars = ax.bar(["Arch 1","Arch 2","Arch 3"], fc, color=colors, width=0.5)
for b, v in zip(bars, fc):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.001,
            f"{v:.4f}", ha="center", va="bottom", fontsize=10, fontweight="bold")
ax.set_title("Coût Final", fontweight="bold")
ax.grid(True, axis="y", linestyle="--", alpha=0.5)

# Accuracy
ax  = axes[2]
x, w = np.arange(3), 0.35
b1 = ax.bar(x-w/2, [results[k]["train_acc"] for k in keys], w, label="Train", color="#2E86C1")
b2 = ax.bar(x+w/2, [results[k]["test_acc"]  for k in keys], w, label="Test",  color="#27AE60")
for b in list(b1)+list(b2):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.05,
            f"{b.get_height():.1f}%", ha="center", va="bottom", fontsize=9, fontweight="bold")
ax.set_xticks(x); ax.set_xticklabels(["Arch 1","Arch 2","Arch 3"])
ax.set_ylim(92, 100); ax.set_title("Accuracy Train vs Test", fontweight="bold")
ax.legend(); ax.grid(True, axis="y", linestyle="--", alpha=0.5)

plt.tight_layout(); plt.show()

## 7. Visualisation des Prédictions

In [ ]:
best_params = results[str([784, 256, 128, 10])]["params"]
preds = predict(X_test, best_params)

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
idx = np.random.choice(X_test.shape[1], 16, replace=False)
for i, j in enumerate(idx):
    ax = axes[i//8][i%8]
    ax.imshow(X_test[:, j].reshape(28, 28), cmap="gray")
    color = "green" if preds[j] == Y_test_raw[j] else "red"
    ax.set_title(f"P:{preds[j]} V:{Y_test_raw[j]}", fontsize=8, color=color)
    ax.axis("off")
plt.suptitle("Prédictions — vert=correct, rouge=erreur", fontsize=10)
plt.tight_layout(); plt.show()

## 8. Résumé Final

In [ ]:
print("=" * 55)
print(f"{'Architecture':<30} {'Train':>8} {'Test':>8}")
print("-" * 55)
for arch in architectures:
    k = str(arch)
    print(f"{k:<30} {results[k]['train_acc']:>7.2f}% {results[k]['test_acc']:>7.2f}%")
print("=" * 55)
best = max(results, key=lambda k: results[k]["test_acc"])
print(f"\n✅ Meilleure architecture : {best}")
print(f"   Test Accuracy          : {results[best]['test_acc']:.2f}%")

In [16]:
pip install keras

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 982.9 kB/s eta 0:00:00eta 0:00:010:01:010m
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 404.6 kB/s eta 0:00:00m eta 0:00:010:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.7/419.7 KB 669.4 kB/s eta 0:00:000:00:010:00:01:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.7/310.7 KB 584.4 kB/s eta 0:00:00m eta 0:00:010:01:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 368.3 kB/s eta 0:00:00m eta 0:00:010:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 KB 330.6 kB/s eta 0:00:000:00:010:00:01:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 KB 285.6 kB/s eta 0:00:001m577.7 kB/s eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.3/87.3 KB 353.8 kB/s eta 0:00:001m480.0 kB/s eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 419.3 kB/s eta 0:00:00m eta 0:00:010:00:01
Note: you may n

In [18]:
# ====================== IMPORTS ======================
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.datasets import mnist

np.random.seed(42)

# ====================== INITIALISATION ======================
def initialize_parameters(layers_dims):
    parameters = {}
    L = len(layers_dims) - 1

    for l in range(1, L + 1):
        n_prev = layers_dims[l-1]
        n_current = layers_dims[l]
        parameters[f"W{l}"] = np.random.randn(n_current, n_prev) * 0.01
        parameters[f"b{l}"] = np.zeros((n_current, 1))

    return parameters


# ====================== ACTIVATIONS ======================
def relu(Z):
    return np.maximum(0, Z)

def relu_derivative(Z):
    return Z > 0

def softmax(Z):
    Z_shift = Z - np.max(Z, axis=0, keepdims=True)
    expZ = np.exp(Z_shift)
    return expZ / np.sum(expZ, axis=0, keepdims=True)


# ====================== FORWARD ======================
def forward_propagation(X, parameters):
    cache = {}
    A = X
    L = len(parameters) // 2

    for l in range(1, L):
        Z = np.dot(parameters[f"W{l}"], A) + parameters[f"b{l}"]
        A = relu(Z)
        cache[f"Z{l}"] = Z
        cache[f"A{l}"] = A

    ZL = np.dot(parameters[f"W{L}"], A) + parameters[f"b{L}"]
    AL = softmax(ZL)

    cache[f"Z{L}"] = ZL
    cache[f"A{L}"] = AL

    return AL, cache


# ====================== COST ======================
def compute_cost(AL, Y):
    m = Y.shape[1]
    cost = -np.sum(Y * np.log(AL + 1e-8)) / m
    return np.squeeze(cost)


# ====================== BACKPROP ======================
def backward_propagation(X, Y, cache, parameters):
    grads = {}
    L = len(parameters) // 2
    m = Y.shape[1]

    AL = cache[f"A{L}"]
    dZ = AL - Y

    for l in reversed(range(1, L + 1)):
        A_prev = X if l == 1 else cache[f"A{l-1}"]

        dW = (1/m) * np.dot(dZ, A_prev.T)
        db = (1/m) * np.sum(dZ, axis=1, keepdims=True)

        grads[f"dW{l}"] = dW
        grads[f"db{l}"] = db

        if l > 1:
            dA_prev = np.dot(parameters[f"W{l}"].T, dZ)
            dZ = dA_prev * relu_derivative(cache[f"Z{l-1}"])

    return grads


# ====================== UPDATE ======================
def update_parameters(parameters, grads, learning_rate):
    L = len(parameters) // 2

    for l in range(1, L + 1):
        parameters[f"W{l}"] -= learning_rate * grads[f"dW{l}"]
        parameters[f"b{l}"] -= learning_rate * grads[f"db{l}"]

    return parameters


# ====================== MODEL ======================
def model(X, Y, layers_dims, num_iterations=500, learning_rate=0.1, print_cost=False):
    costs = []
    parameters = initialize_parameters(layers_dims)

    for i in range(num_iterations):
        AL, cache = forward_propagation(X, parameters)
        cost = compute_cost(AL, Y)
        grads = backward_propagation(X, Y, cache, parameters)
        parameters = update_parameters(parameters, grads, learning_rate)

        if i % 100 == 0:
            costs.append(cost)

        if print_cost and i % 200 == 0:
            print(f"Iteration {i} | Cost = {cost:.4f}")

    return parameters, costs


# ====================== UTILS ======================
def predict(X, parameters):
    AL, _ = forward_propagation(X, parameters)
    return np.argmax(AL, axis=0)

def one_hot(Y, num_classes=10):
    return np.eye(num_classes)[Y].T


# ====================== DATA ======================
(X_train_raw, Y_train_raw), (X_test_raw, Y_test_raw) = mnist.load_data()

X_train = X_train_raw.reshape(X_train_raw.shape[0], -1).T / 255.0
X_test = X_test_raw.reshape(X_test_raw.shape[0], -1).T / 255.0

Y_train = one_hot(Y_train_raw)
Y_test = one_hot(Y_test_raw)

print("X_train:", X_train.shape)
print("Y_train:", Y_train.shape)


# ====================== ARCHITECTURE 1 ======================
layers1 = [784, 128, 10]
params1, costs1 = model(X_train, Y_train, layers1, print_cost=True)

pred1 = predict(X_test, params1)
acc1 = np.mean(pred1 == Y_test_raw) * 100
print("Accuracy 1:", acc1)


# ====================== ARCHITECTURE 2 ======================
layers2 = [784, 256, 128, 10]
params2, costs2 = model(X_train, Y_train, layers2, print_cost=True)

pred2 = predict(X_test, params2)
acc2 = np.mean(pred2 == Y_test_raw) * 100
print("Accuracy 2:", acc2)


# ====================== ARCHITECTURE 3 ======================
layers3 = [784, 512, 256, 128, 10]
params3, costs3 = model(X_train, Y_train, layers3, print_cost=True)

pred3 = predict(X_test, params3)
acc3 = np.mean(pred3 == Y_test_raw) * 100
print("Accuracy 3:", acc3)


# ====================== PLOT ======================
plt.plot(costs1, label="Arch1")
plt.plot(costs2, label="Arch2")
plt.plot(costs3, label="Arch3")

plt.xlabel("Iterations (x100)")
plt.ylabel("Cost")
plt.title("Cost Evolution")
plt.legend()
plt.show()


# ====================== RESULT ======================
print("Final Results:")
print("Arch1:", acc1)
print("Arch2:", acc2)
print("Arch3:", acc3)

ModuleNotFoundError: No module named 'tensorflow'